# `RUN__pdf_ocr` — the CafeF filing parse, on whatever GPU is available

Runs the production OCR cascade over one ticker's filings and writes a **run folder**, not a
statement CSV. Two places it is meant to run and they are the same code:

* **here**, against `raw_data/cafef/` and this machine's RTX 3050 (4 GiB);
* **on a Kaggle T4** (15 GiB), through
  `cd src\\kaggle_gpu && python -m kgpu run pdf-ocr` — the payload ships the filings, the
  twelve charts of accounts, the statement CSVs already on disk and both OCR models, and
  `kgpu_bootstrap` points this notebook at them.

## ⚠️ What it does NOT do

1. **It writes no statement CSV.** Merging a recovered quarter back into
   `raw_data/cafef/financials/statements/` stays a deliberate act through Dagster, with a
   pre-run backup and a diff of every column — this repo has measured four separate runs in
   which a `periods` build silently DOWNGRADED a quarter it was given only for history, while
   the log said `RUN_SUCCESS` (CLAUDE.md §6-2-vicies, §6-2-unvicies, §6-2-quatervicies,
   §6-2-quinvicies).
2. **It does not retry an ALTERNATE filing.** `build()` falls back to a second filing of the
   same period when a statement is absent; `_parse_cascaded` — which is what runs here — does
   not, so a quarter that `build()` would recover from its alternate reads `absent` here. Name
   the period explicitly to parse it.
3. **It does not de-cumulate.** An annual or semi-annual filing's income statement is the year
   to date, and turning it into the quarter needs Q1..Q(q-1) of the same year. A one-document
   run does not have them, so `compare()` REFUSES to score a cumulative P&L rather than
   reporting every cell as changed.

## ⚠️ And a run on another machine is a different PROCEDURE

Kaggle ships its own torch and onnxruntime, and DB detection under `CUDAExecutionProvider` is
not bit-identical to detection on the CPU wheel. So the run scores itself: `compare()` reads
every parsed cell against the row already on disk and the last cell prints the verdict. That
comparison is the measurement — nothing here assumes a T4 reproduces this machine.

In [ ]:
# ── PARAMETERS ────────────────────────────────────────────────────────────────
# ⚠️ Rewritten IN PLACE by `kgpu build` (src/kaggle_gpu/kaggle_config.json), so every name
# below must stay a TOP-LEVEL assignment. A parameter kgpu cannot find refuses the push.
EXCHANGE = "HOSE"
SYMBOL = "VCB"
PERIODS = ["Q1-2026"]          # None = every period this ticker files. A period it does not file RAISES.
ALLOW_PARENT = False           # fall back to the STANDALONE filing where no consolidated one exists
LAYERS = None                  # None = the full 47-layer cascade; else a list of layer names, in cascade order
COMPARE = True                 # score every parsed cell against the statement CSV on disk
OUT_ROOT = None                # None = <repo>/reports/pdf_ocr, anchored to the module file rather than the CWD
NOTES = ""

# ⚠️ **NOT A KNOB: the engine.** Every `ParseLayer` names its own engine and DPI, so the
# cascade decides what runs — `onnx` for 43 of the 47 layers, `tesseract` for the four that
# are skipped wherever it is not installed (a Kaggle worker, for one).


In [ ]:
# ── DEPENDENCIES ──────────────────────────────────────────────────────────────
# ⚠️ **THE KAGGLE IMAGE HAS TORCH AND OPENCV AND NOT THE OCR STACK.** VietOCR, PyMuPDF and
# pyclipper are not on it, so a job with `enable_internet: false` cannot run this notebook at
# all — the models travel in the payload, the PACKAGES cannot. Each import is probed and only
# what is missing is installed, so a machine that already has them (this one) does nothing.
#
# ⚠️ `vietocr` is installed with `--no-deps` DELIBERATELY. Its metadata pins `pillow==10.2.0`
# and `albumentations==1.4.2`, which pip would happily downgrade a whole image to satisfy —
# and the predictor path needs none of them: torch, torchvision, numpy, PIL, yaml, einops,
# gdown, tqdm and requests, every one of which Kaggle already ships.
import importlib.util
import subprocess
import sys

_NEEDED = [
    ("fitz", "pymupdf", []),                 # the PDF reader the parser opens every filing with
    ("pyclipper", "pyclipper", []),          # DeepDoc's DB post-processing
    ("shapely", "shapely", []),
    ("einops", "einops", []),
    ("vietocr", "vietocr", ["--no-deps"]),   # see above
    # ⚠️ **PINNED, AND THE PIN IS THE WHOLE POINT — measured on Kaggle 2026-08-28.**
    # A bare `onnxruntime-gpu` resolved to 1.29.0, which requires **cuDNN 9 with CUDA 13**;
    # Kaggle's image is CUDA 12.8, so `CUDAExecutionProvider` FAILED TO LOAD and DB detection
    # ran on the worker's CPU. The run was correct and slower, and the only trace was a
    # warning inside a wall of ANSI-coloured onnxruntime noise. 1.19-1.22 is the CUDA 12 /
    # cuDNN 9 line.
    # ⚠️ A RANGE AND NOT THIS REPO'S OWN 1.20.1: that exact version is not published for
    # Kaggle's cp312 Linux (the index offers 1.20.0 and 1.20.2 and no 1.20.1), so an `==` pin
    # fails the INSTALL outright — measured, and it cost the second run.  Pin the LINE.
    ("onnxruntime", "onnxruntime-gpu>=1.19,<1.23", []),
]

_missing = [(mod, pkg, flags) for mod, pkg, flags in _NEEDED
            if importlib.util.find_spec(mod) is None]
print("missing:", [m for m, _, _ in _missing] or "nothing — the stack is already here")
for _mod, _pkg, _flags in _missing:
    print(f"  pip install {_pkg} {' '.join(_flags)}", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_flags, _pkg], check=True)

still = [m for m, _, _ in _NEEDED if importlib.util.find_spec(m) is None]
if still:
    raise RuntimeError(
        f"{still} could not be imported after the install step. On Kaggle that means the "
        f"kernel has no internet — set \"enable_internet\": true on the job. The MODELS ship "
        f"in the payload; the packages cannot."
    )


In [ ]:
# ── WHERE THE REPO AND THE DATA ARE ───────────────────────────────────────────
# On a worker `kgpu_bootstrap` has already unpacked the source to /kaggle/working/src, put it
# on sys.path and set CAFEF_DATA_ROOT / CAFEF_MODELS_DIR at the shipped payload. Locally this
# notebook sits IN the repo, so walk up for `src` and let the module defaults stand.
import os
import sys
from pathlib import Path

if importlib.util.find_spec("web_scraper") is None:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "web_scraper").is_dir():
            sys.path.insert(0, str(candidate / "src"))
            break
    else:
        raise RuntimeError(f"no src/web_scraper above {here}")

from web_scraper import pdf_ocr_job as job
from web_scraper.cafef_financials import FinancialsBuilder

DATA_ROOT = job.use_data_root()          # $CAFEF_DATA_ROOT on a worker, raw_data/cafef here
MODELS = job.use_models()                # $CAFEF_MODELS_DIR on a worker, src/web_scraper/models here
print(f"data root : {DATA_ROOT}")
print(f"models    : {MODELS}")

# ⚠️ **AN ABSENT CHECKPOINT IS A DOWNLOAD, NOT AN ERROR — so it is turned into one here.**
# Without a local `vgg_seq2seq.pth` vietocr fetches ~90 MB from vocr.vn on the first page,
# which fails outright on a kernel with no internet and silently costs a cold start on one
# with it. The payload ships both models; this says so out loud when it did not.
if MODELS.get("vietocr") is None:
    print("WARNING: no local VietOCR checkpoint — the recogniser will try to DOWNLOAD one.")
if MODELS.get("det") is None:
    print("WARNING: no local DeepDoc detector — it will be fetched from HuggingFace.")

import torch
print(f"torch     : {torch.__version__}  cuda={torch.cuda.is_available()}"
      + (f"  {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else ""))
# ⚠️ **`get_available_providers()` IS AN ADVERTISEMENT.** On the first Kaggle run it listed
# CUDAExecutionProvider while the session could not create one. `engine_report()` builds the
# detector and reads back what the SESSION holds, which is the only honest answer — and it goes
# into the run's metadata.json so a later reader can tell a GPU-detected run from a CPU one.
ENGINE = job.engine_report()
print(f"onnxrt    : {ENGINE.get('onnxruntime')}")
print(f"  advertises  : {ENGINE.get('onnxruntime_advertises')}")
print(f"  DETECTION on: {ENGINE.get('det_providers')}")
print(f"  RECOGNITION : {ENGINE.get('recognizer_device')}")
if "CUDAExecutionProvider" not in (ENGINE.get("det_providers") or []):
    print("WARNING: DB detection is on the CPU (~1.8 s/page against ~0.25 on a GPU). The run "
          "is still correct; it is just paying for the wrong half of the machine.")


In [ ]:
# ── WHICH FILINGS ─────────────────────────────────────────────────────────────
# `documents()` chooses them — consolidated over standalone, the audited annual standing in for
# Q4 — and this only filters to PERIODS. The choice is never re-implemented here: it carries a
# measured guard against an annual report changing the ENTITY of a Q4 row.
builder = FinancialsBuilder(logger=None)
tasks = job.plan(builder, EXCHANGE, SYMBOL, periods=PERIODS, allow_parent=ALLOW_PARENT)

for t in tasks:
    size = os.path.getsize(t.path) / 1024 ** 2
    print(f"{t.period:<8} {t.file:<60} {size:>6.1f} MB  "
          f"consolidated={t.consolidated} {t.assurance}"
          + ("  CUMULATIVE" if t.cumulative else ""))
print(f"\n{len(tasks)} filing(s); cascade = "
      f"{len(job.select_layers(LAYERS))} of {len(FinancialsBuilder.LAYERS)} layers")


In [ ]:
# ── THE PARSE ─────────────────────────────────────────────────────────────────
# ⚠️ Each document's JSON is written BEFORE the next one starts (§5 rule 20): this stage has
# single documents that cost over an hour, and a run that holds its results in memory loses
# every one of them to the first crash. Expect long silences between the per-document lines —
# the cascade prints only when a statement is accepted or a filing is finished.
folder = job.run(
    tasks,
    out_root=OUT_ROOT or job.DEFAULT_OUT_ROOT,
    layers=LAYERS,
    notes=NOTES,
    compare_with_disk=COMPARE,
)
print(folder)


In [ ]:
# ── WHAT CAME OUT ─────────────────────────────────────────────────────────────
# The run folder is the artefact; this reads it back rather than re-using anything in memory,
# so the cell measures what a later reader would actually get.
import json

import pandas as pd

metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
print(f"run   : {metadata['run_id']}")
print(f"commit: {metadata['git_commit']}")
print(f"gpu   : {metadata['environment']['gpu'].get('name')}")
print(f"took  : {metadata['execution']['runtime']}")

for path in sorted((folder / "documents").glob("*.json")):
    doc = json.loads(path.read_text(encoding="utf-8"))
    print(f"\n{doc['period']}  {doc['document']}  ({doc['seconds'] / 60:.1f} min)")
    for report, entry in (doc.get("compare") or {}).items():
        got = doc["accepted"].get(report)
        head = f"  {report:<18} {entry['verdict']}"
        if got and "identical" in entry:
            head += (f"  {entry['identical']}/{entry['cells_run']} cells identical"
                     f", {len(entry['changed'])} changed"
                     f", layer {entry['run_layer']} vs {entry['disk_layer']} on disk")
        elif got:
            head += f"  [{got['layer']}] {got['items']} items"
        print(head)
        for column, (on_disk, in_run) in list((entry.get("changed") or {}).items())[:5]:
            print(f"      {column}: disk {on_disk:,} -> run {in_run:,}")

summary = pd.DataFrame(metadata["results"])
summary
